### Required libraries

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

### Main search function

In [9]:
def search_gdelt(query, start_date, end_date, max_records=250):
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    params = {
        "query": query,
        "mode": "artlist",
        "maxrecords": max_records,
        "startdatetime": start_date.strftime("%Y%m%d%H%M%S"),
        "enddatetime": end_date.strftime("%Y%m%d%H%M%S"),
        "format": "json"
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    response = requests.get(url, params=params, headers=headers, timeout=30)
    return response.json()

### Download by range

In [10]:
def download_gdelt_range(query, start_date, end_date, sleep_seconds=6):
    all_articles = []
    current = start_date
    
    while current < end_date:
        week_end = min(current + timedelta(days=7), end_date)
        try:
            results = search_gdelt(query, current, week_end)
            articles = results.get("articles", [])
            for a in articles:
                a['week_start'] = current.strftime("%Y-%m-%d")
            all_articles.extend(articles)
            print(f"{current.date()} → {week_end.date()}: {len(articles)} artículos")
        except Exception as e:
            print(f"{current.date()} → {week_end.date()}: ERROR — {str(e)[:50]}")
        
        current = week_end
        time.sleep(sleep_seconds)
    
    return pd.DataFrame(all_articles)


### Weekly headlines

Articles are capped by GDELT to get the body, so for now we're sticking to get only headers and feed the NLP with that. In the future I expect to change this and be able to feed the NLP with the news first 2-3 paragraphs as well.

In [11]:
df_news = download_gdelt_range(
    query="crude oil WTI price",
    start_date=datetime(2024, 1, 1),
    end_date=datetime(2026, 3, 1)
)

2024-01-01 → 2024-01-08: 101 artículos
2024-01-08 → 2024-01-15: ERROR — HTTPSConnectionPool(host='api.gdeltproject.org', p
2024-01-15 → 2024-01-22: ERROR — HTTPSConnectionPool(host='api.gdeltproject.org', p
2024-01-22 → 2024-01-29: ERROR — HTTPSConnectionPool(host='api.gdeltproject.org', p
2024-01-29 → 2024-02-05: ERROR — HTTPSConnectionPool(host='api.gdeltproject.org', p
2024-02-05 → 2024-02-12: 84 artículos
2024-02-12 → 2024-02-19: ERROR — HTTPSConnectionPool(host='api.gdeltproject.org', p
2024-02-19 → 2024-02-26: 166 artículos
2024-02-26 → 2024-03-04: 173 artículos
2024-03-04 → 2024-03-11: ERROR — HTTPSConnectionPool(host='api.gdeltproject.org', p
2024-03-11 → 2024-03-18: ERROR — HTTPSConnectionPool(host='api.gdeltproject.org', p
2024-03-18 → 2024-03-25: ERROR — HTTPSConnectionPool(host='api.gdeltproject.org', p
2024-03-25 → 2024-04-01: 137 artículos
2024-04-01 → 2024-04-08: ERROR — HTTPSConnectionPool(host='api.gdeltproject.org', p
2024-04-08 → 2024-04-15: ERROR — HTTPSConnectionPo

In [12]:
print(f"Total artículos: {len(df_news)}")
print(f"Semanas cubiertas: {df_news['week_start'].nunique()}")
print(f"\nIdiomas:\n{df_news['language'].value_counts().head()}")
print(f"\nDominios más frecuentes:\n{df_news['domain'].value_counts().head(10)}")
print(df_news[['title', 'seendate', 'domain']].head(5))


df_news.to_csv("../01_data/raw/news/gdelt_wti_raw.csv", index=False)
print("Guardado en 01_data/raw/news/gdelt_wti_raw.csv")

Total artículos: 4599
Semanas cubiertas: 44

Idiomas:
language
English      3843
Spanish       309
Polish        106
Hungarian     103
Romanian       80
Name: count, dtype: int64

Dominios más frecuentes:
domain
marketscreener.com          374
zerohedge.com               203
fxstreet.com                169
finance.yahoo.com           130
fool.com.au                 124
seekingalpha.com            102
oilprice.com                 90
hellenicshippingnews.com     81
rttnews.com                  71
theglobeandmail.com          68
Name: count, dtype: int64
                                               title          seendate  \
0             Oil prices today : WTI prices are down  20240105T160000Z   
1                   WTI Crude Oil Technical Analysis  20240104T094500Z   
2  Baryłka ropy Brent podrożała w środę rano do 7...  20240103T073000Z   
3  WTI extends gains to near $73 . 00 on Israel -...  20240104T050000Z   
4  Where Will WTI Crude Oil Price be at End - 202...  20240102T141500Z  

### News cleaning

The models are trained mainly english, and although they support other languages we believe the sentiment accuracy is higher in english, and the material is more abundant. 

- Deleting news in other languages other than english.
- Deleting repeated articles.
- Some soures are heavily biased, those are being removed as well since they affect the sentiment detected.

In [13]:
# Celda 7 — Limpieza
df_clean = df_news.copy()

# Solo inglés
df_clean = df_clean[df_clean['language'] == 'English']

# Eliminar duplicados por título
df_clean = df_clean.drop_duplicates(subset='title')

# Parsear fecha correctamente
df_clean['datetime'] = pd.to_datetime(df_clean['seendate'], format='%Y%m%dT%H%M%SZ', utc=True)
df_clean = df_clean.sort_values('datetime').reset_index(drop=True)

# Quedarse solo con columnas útiles
df_clean = df_clean[['title', 'datetime', 'domain', 'url']]

print(f"Artículos después de limpieza: {len(df_clean)}")
print(f"Rango: {df_clean['datetime'].min()} a {df_clean['datetime'].max()}")
print(df_clean.head(5))

# Guardar
df_clean.to_csv("../01_data/processed/gdelt_wti_clean.csv", index=False)
print("Guardado en 01_data/processed/gdelt_wti_clean.csv")

Artículos después de limpieza: 3290
Rango: 2024-01-01 00:30:00+00:00 a 2026-02-22 11:15:00+00:00
                                               title  \
0                2023 Was a Bad Year for Commodities   
1  Global oil market  poised for significant shif...   
2        5 things to watch on the ASX 200 on Tuesday   
3  Asia Shares Get 2024 to Steady Start With Busy...   
4  oil prices : Oil jumps 1 % in New Year after U...   

                   datetime                        domain  \
0 2024-01-01 00:30:00+00:00                  oilprice.com   
1 2024-01-01 05:00:00+00:00                     zawya.com   
2 2024-01-01 22:00:00+00:00                   fool.com.au   
3 2024-01-02 04:00:00+00:00              money.usnews.com   
4 2024-01-02 04:00:00+00:00  economictimes.indiatimes.com   

                                                 url  
0  https://oilprice.com/Energy/Energy-General/202...  
1  https://www.zawya.com/en/markets/commodities/g...  
2  https://www.fool.com.au/2024/01